In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from pathlib import Path
from collections import defaultdict

plt.style.use("ggplot")

candidates = [
    Path("data") / "2019-Nov.csv",
    Path.cwd() / "data" / "2019-Nov.csv",
    Path.cwd() / ".." / "data" / "2019-Nov.csv",
    Path.cwd() / "notebook" / ".." / "data" / "2019-Nov.csv",
]
csv_path = next((p.resolve() for p in candidates if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError(
        "Could not find data/2019-Nov.csv. Checked: "
        + ", ".join(str(p.resolve()) for p in candidates)
    )

print(f"Loading data from: {csv_path}")
file_size_gb = csv_path.stat().st_size / 1024 ** 3
print(f"CSV file size: {file_size_gb:.2f} GB")

parse_dates = ["event_time"]
dtype = {
    "event_type": "category",
    "product_id": "int64",
    "category_id": "int64",
    "category_code": "string",
    "brand": "string",
    "price": "float32",
    "user_id": "int64",
    "user_session": "string",
}


def safe_rate(numerator, denominator):
    return (numerator / denominator * 100) if denominator else np.nan


def aggregate_counts(target, values):
    for key, value in values.items():
        target[key] += int(value)


if file_size_gb > 3:
    print("Large dataset detected. Processing in chunks.")
    reader = pd.read_csv(
        csv_path,
        parse_dates=parse_dates,
        dtype=dtype,
        chunksize=250_000,
        low_memory=False,
    )

    event_counts = defaultdict(int)
    hour_counts = np.zeros(24, dtype="int64")
    brand_revenue = defaultdict(float)
    category_revenue = defaultdict(float)
    total_revenue = 0.0
    total_rows = 0

    for chunk_index, chunk in enumerate(reader, start=1):
        total_rows += len(chunk)
        aggregate_counts(event_counts, chunk["event_type"].value_counts().to_dict())

        if chunk["event_time"].dtype == object:
            chunk["event_time"] = pd.to_datetime(chunk["event_time"], errors="coerce")

        purchases = chunk[chunk["event_type"] == "purchase"].copy()
        if not purchases.empty:
            total_revenue += purchases["price"].sum(skipna=True)
            purchases["brand"] = purchases["brand"].fillna("Unknown")
            purchases["category_code"] = purchases["category_code"].fillna("Unknown")
            purchases["hour"] = purchases["event_time"].dt.hour

            for brand, rev in purchases.groupby("brand")["price"].sum().items():
                brand_revenue[brand] += float(rev)

            for category, rev in purchases.groupby("category_code")["price"].sum().items():
                category_revenue[category] += float(rev)

            hour_series = (
                purchases["hour"].dropna().astype(int).value_counts().reindex(range(24), fill_value=0)
            )
            hour_counts += hour_series.to_numpy(dtype="int64")

        print(f"Processed chunk {chunk_index}: {len(chunk):,} rows (total {total_rows:,})")

    views = event_counts.get("view", 0)
    carts = event_counts.get("cart", 0)
    purchases = event_counts.get("purchase", 0)

else:
    df = pd.read_csv(csv_path, parse_dates=parse_dates, dtype=dtype, low_memory=False)
    print("Loaded full dataset:", df.shape)
    if df["event_time"].dtype == object:
        df["event_time"] = pd.to_datetime(df["event_time"], errors="coerce")

    df["hour"] = df["event_time"].dt.hour
    event_counts = df["event_type"].value_counts()
    views = int(event_counts.get("view", 0))
    carts = int(event_counts.get("cart", 0))
    purchases = int(event_counts.get("purchase", 0))

    purchase_df = df[df["event_type"] == "purchase"].copy()
    total_revenue = float(purchase_df["price"].sum(skipna=True))
    purchase_df["brand"] = purchase_df["brand"].fillna("Unknown")
    purchase_df["category_code"] = purchase_df["category_code"].fillna("Unknown")

    brand_revenue = purchase_df.groupby("brand")["price"].sum().to_dict()
    category_revenue = purchase_df.groupby("category_code")["price"].sum().to_dict()
    hour_counts = (
        purchase_df["hour"].dropna().astype(int).value_counts().reindex(range(24), fill_value=0).to_numpy(dtype="int64")
    )

view_to_cart = safe_rate(carts, views)
cart_to_purchase = safe_rate(purchases, carts)
overall_conversion = safe_rate(purchases, views)
view_dropoff = 100 - view_to_cart if not np.isnan(view_to_cart) else np.nan
cart_dropoff = 100 - cart_to_purchase if not np.isnan(cart_to_purchase) else np.nan

print("Summary")
print("-------")
print(f"Views: {views:,}")
print(f"Carts: {carts:,}")
print(f"Purchases: {purchases:,}")
print(f"View → Cart: {view_to_cart:.2f}%")
print(f"Cart → Purchase: {cart_to_purchase:.2f}%")
print(f"Overall Conversion: {overall_conversion:.2f}%")
print(f"View Drop-Off: {view_dropoff:.2f}%")
print(f"Cart Drop-Off: {cart_dropoff:.2f}%")
print(f"Total Revenue: ${total_revenue:,.2f}")

brand_series = pd.Series(brand_revenue).sort_values(ascending=False).head(10)
category_series = pd.Series(category_revenue).sort_values(ascending=False).head(10)

print("\nTop 10 brands by purchase revenue:")
print(brand_series)
print("\nTop 10 category codes by purchase revenue:")
print(category_series)

fig = go.Figure(
    go.Funnel(
        y=["Views", "Cart", "Purchase"],
        x=[views, carts, purchases],
        textinfo="value+percent initial",
    )
)
fig.update_layout(title_text="Marketing Funnel")

try:
    fig.show()
except Exception as exc:
    print(f"Plotly render failed: {exc}")
    output_file = Path("funnel_plot.html").resolve()
    fig.write_html(output_file, include_plotlyjs="cdn")
    print(f"Saved funnel plot HTML to: {output_file}")

hour_series = pd.Series(hour_counts, index=pd.RangeIndex(24, name="hour"))
plt.figure(figsize=(10, 5))
sns.lineplot(x=hour_series.index, y=hour_series.values, marker="o")
plt.title("Purchases by Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Purchase Count")
plt.xticks(range(24))
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


Loading data from: G:\Marketing_Funnel_Analysis\data\2019-Nov.csv
CSV file size: 8.39 GB
Large dataset detected. Reading in chunks to avoid MemoryError.


ParserError: Error tokenizing data. C error: out of memory